In [50]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
import tensorflow as tf
import tensorflow_probability as tfp


import re
import os
import copy
import warnings
import functools
import contextlib
import time

from scipy import stats
from decimal import Decimal
from datetime import datetime
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks
from bayes_opt import BayesianOptimization
from sklearn.metrics import confusion_matrix, classification_report, fbeta_score
from itertools import product


In [51]:
plt.style.use('dark_background')
pd.set_option("display.precision", 2)

tf.get_logger().setLevel('INFO')

seed = 1
tf.keras.utils.set_random_seed(seed)
tf.random.set_seed(seed)
tf.config.experimental.enable_op_determinism()
np.random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

In [52]:
# Data Loading
df = pd.read_csv('data/data_incentive_attitudes_residencedata.csv')

# Some basic data preprocessing steps
df = df.drop(['X', 'trip','value','long_o', 'lat_o', 'long_d.x', 
              'lat_d.x', 'pincode', 'address','Prefecture', 'City', 
              'Town','lat_d.return','long_d.return', 'dest_inside'], axis='columns')
df = df.rename(columns={'pubcost':'traincost', 
                   'pubtime':'traintime', 
                   'pubavail':'trainavail',
                    'bicycleincentive':'bikeincentive'})

# Replacing 'pub' with 'train' in `mode` column.
selected_mode = df['mode'].to_numpy()
df['mode'] = np.where(selected_mode=='pub', 'train', selected_mode)

purpose_code_dict = {100: 'Commuting to work / school',
                    101: 'Go Home',
                    200: 'Shopping for daily necessities',
                    201: 'Shopping other than daily necessities',
                    202: 'Meals and entertainment',
                    300: 'business',
                    400: 'Outpatient',
                    500: 'Pick-up and drop-off',
                    600: 'Sightseeing / Leisure',
                    998: 'others'}

attitudinal_variables = [x for x in df.columns if bool(re.search(r'\d', x))]
ohe_generic_variables = dict()

df['walkavail'] = np.where(df['mode']=='walk', 1, np.where(df['cardistance']<=7, 1, 0))
df['busincentive'] = np.where(df['incentivezone']==1, df['busincentive'], 0)
df['carincentive'] = np.where(df['incentivezone']==1, df['carincentive'], 0)
df['trainincentive'] = np.where(df['incentivezone']==1, df['trainincentive'], 0)
df['bikeincentive'] = np.where(df['incentivezone']==1, df['bikeincentive'], 0)
df['walkincentive'] = np.where(df['incentivezone']==1, df['walkincentive'], 0)
df['motorincentive'] = np.where(df['incentivezone']==1, df['motorincentive'], 0)


In [53]:
# utility functions for extracting time-zones and trip duration information from arrival and departure times of a trip.

def get_trip_duration(df):
    """
    Returns the total minutes a trip lasted given its departure and arrival datetimes.
    """
    dep_datetime = datetime.strptime(df['departure_time'], '%m/%d/%Y %H:%M')
    arr_datetime = datetime.strptime(df['arrival_time'], '%m/%d/%Y %H:%M')
    
    return int((arr_datetime - dep_datetime).total_seconds()/60)

def get_day_zones(trip_datetime):
    """
    Divides the time into 4 zones and ordinally encoding them.
    Early morning (00:00 to 6:00) - 1
    AM peak (6:00 to 10:00) - 2
    Off peak (10:00 to 16:00) - 3
    PM peak (16:00 to 20:00) - 4
    Evening (20:00 to 00:00) - 5
    """
    dep_hrs, dep_mins = [int(val) for val in trip_datetime.split(' ')[1].split(':')]
    if dep_hrs >= 0 and dep_hrs < 6:
        return 1
    elif dep_hrs >= 6 and dep_hrs < 10:
        return 2
    elif dep_hrs >= 10 and dep_hrs < 16:
        return 3
    elif dep_hrs >= 16 and dep_hrs < 20:
        return 4
    else:
        return 5

ohe = OneHotEncoder()
df['high_income'] = np.where(df['INCOME'] >= 5, 1, 0)

# `job_type` column
job_type_sparse_matrix = ohe.fit_transform(df['job_type'].to_numpy().reshape(-1, 1)).toarray()
job_type_column_names = ['job_type_'+job for job in df['job_type'].unique()]
job_type_df = pd.DataFrame(data=job_type_sparse_matrix, columns=job_type_column_names)
job_type_df = job_type_df.drop(job_type_df.columns[0], axis=1)
ohe_generic_variables['jobs'] = list(job_type_df.columns)

# `information` column
info_sparse_matrix = ohe.fit_transform(df['information'].to_numpy().reshape(-1, 1)).toarray()
info_df = pd.DataFrame(data=info_sparse_matrix, columns=ohe.categories_[0])
info_df = info_df.drop(info_df.columns[0], axis=1)
ohe_generic_variables['info'] = list(info_df.columns)

# `SEX` column
SEX_sparse_matrix = ohe.fit_transform(df['SEX'].to_numpy().reshape(-1, 1)).toarray()
SEX_df = pd.DataFrame(data=SEX_sparse_matrix, columns=['Male', 'Female'])
SEX_df = SEX_df.drop(SEX_df.columns[0], axis=1)

# `Purpose` column
Purpose_sparse_matrix = ohe.fit_transform(df['Purpose'].to_numpy().reshape(-1, 1)).toarray()
Purpose_column_names = ['Purpose_'+str(purpose_code) for purpose_code in df['Purpose'].unique()]
Purpose_df = pd.DataFrame(data=Purpose_sparse_matrix, columns=Purpose_column_names)
Purpose_df = Purpose_df.drop('Purpose_999', axis=1)
ohe_generic_variables['purposes'] = list(Purpose_df.columns)

# Getting the trip duration in minutes
df['trip_duration'] = df[['departure_time', 'arrival_time']].T.apply(get_trip_duration)

# Converting departure time in time zones and one-hot encoding it
dep_time_zones_sparse_matrix = ohe.fit_transform(df['departure_time'].apply(get_day_zones).to_numpy().reshape(-1, 1)).toarray()
dep_time_zones_names = ['Early_morning_departure', 'AM_peak_departure', 'Off_peak_departure', 'PM_peak_departure', 'Night_departure']
dep_time_zones_df = pd.DataFrame(data=dep_time_zones_sparse_matrix, columns=dep_time_zones_names)
dep_time_zones_df = dep_time_zones_df.drop('Night_departure', axis=1)
ohe_generic_variables['departures'] = list(dep_time_zones_df.columns)

# combining all the one-hot encoded dataframes with the main dataframe
df2 = pd.concat([df, dep_time_zones_df, Purpose_df, SEX_df, job_type_df, info_df], axis=1).drop(
    ['departure_time', 'arrival_time', 'Purpose', 'SEX', 'job_type'], axis='columns')

# Removing some unnecessary columns
df2 = df2.drop(['user_id', 'trip_id', 'recco', 'information', 'incentivezone', 'task', 'income_con'], axis='columns')

df2.head()

,buscost,bustime,carcost,cartime,cardistance,traincost,traintime,walktime,walkcost,biketime,...,job_type_Part time job,job_type_Management executive,job_type_civil servant,job_type_Housewife,job_type_Self employed/ Freelance,job_type_Others,job_type_Unemployed,no info,only enviro,only health
0,770,69.4,151.7,43.72,15.17,240,60.9,182.04,0,60.68,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,690,66.2,128.6,38.96,12.86,240,44.9,154.32,0,51.44,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,770,69.4,151.7,43.72,15.17,240,60.9,182.04,0,60.68,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
3,500,56.2,118.5,29.53,11.85,400,59.0,142.20,0,47.40,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
4,770,69.4,151.7,43.72,15.17,240,60.9,182.04,0,60.68,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0


## Train and Test Split

In [54]:
def get_CBD_data_split(df, drop_att_vars=True, apply_smote=False, apply_tomek_links=False,
                      apply_standardization=False, cols_to_standardize=[]):
    if drop_att_vars:
        df = df.drop(attitudinal_variables, axis='columns')
        
        oCBD_df = df.loc[df['address_inside'] == False]
        iCBD_df = df.loc[(df['address_inside'] == True)]
        
        label_mapping_dict = {'bus': 1, 'car': 2, 'train': 3, 'walk': 4, 'bike': 5, 'motor': 6}
                
        y_oCBD = np.array([label_mapping_dict[label] for label in oCBD_df['mode']])
        X_oCBD = oCBD_df.drop('mode', axis='columns')
        y_iCBD = np.array([label_mapping_dict[label] for label in iCBD_df['mode']])
        X_iCBD = iCBD_df.drop('mode', axis='columns')
        
        if apply_smote:
            smote = SMOTE(random_state=500)
            X_oCBD, y_oCBD = smote.fit_resample(X_oCBD, y_oCBD)
            X_iCBD, y_iCBD = smote.fit_resample(X_iCBD, y_iCBD)
        
        if apply_tomek_links:
            tomek = TomekLinks()
            X_oCBD, y_oCBD = tomek.fit_resample(X_oCBD, y_oCBD)
            X_iCBD, y_iCBD = tomek.fit_resample(X_iCBD, y_iCBD)
        
        if apply_standardization:
            ss =  StandardScaler()
            X_oCBD_scaled_arr = ss.fit_transform(X_oCBD[cols_to_standardize])
            X_iCBD_scaled_arr = ss.transform(X_iCBD[cols_to_standardize])
            X_oCBD_scaled = pd.DataFrame(X_oCBD_scaled_arr, columns=cols_to_standardize)
            X_iCBD_scaled = pd.DataFrame(X_iCBD_scaled_arr, columns=cols_to_standardize)
            X_oCBD_scaled.set_index(X_oCBD.index, inplace=True)
            X_iCBD_scaled.set_index(X_iCBD.index, inplace=True)
            X_oCBD.iloc[0:X_oCBD.shape[0], X_oCBD.columns.get_indexer(cols_to_standardize)] = X_oCBD_scaled
            X_iCBD.iloc[0:X_iCBD.shape[0], X_iCBD.columns.get_indexer(cols_to_standardize)] = X_iCBD_scaled
        
        return ((X_oCBD, y_oCBD), (X_iCBD, y_iCBD)), label_mapping_dict
    

In [55]:
output_categories = ['bus', 'car', 'train', 'walk', 'bike', 'motor']

asc_variable_names = {
    'bus': ['buscost', 'bustime'],
    'car': ['carcost', 'cartime'],
    'train': ['traincost', 'traintime'], 
    'walk': ['walkcost', 'walktime'],
    'bike': [ 'bikecost', 'biketime'],
    'motor': ['motorcost', 'motortime']
}

avail_variable_names = {
    'bus': 'busavail',
    'train': 'trainavail',
    'walk': 'walkavail',
    'bike': 'bikeavail',
    'motor': 'motoravail'
}

In [56]:
((X_oCBD, y_oCBD), (X_iCBD, y_iCBD)), label_mapping_dict = get_CBD_data_split(df2, drop_att_vars=True)

In [57]:
print("Main model training data (outside CBD):", y_oCBD.shape[0])
print("Target region data (inside CBD):", y_iCBD.shape[0])

Main model training data (outside CBD): 1851
Target region data (inside CBD): 1235


In [58]:
avail_variable_indexer = dict(zip(avail_variable_names.keys(), 
                                  X_oCBD.columns.get_indexer(avail_variable_names.values())))
avail_variable_indexer

{'bus': 19, 'train': 20, 'walk': 28, 'bike': 21, 'motor': 22}

# <ins><b>Model Formulation</b></ins>

In [59]:
# Helper functions

def make_val_and_grad_fn(value_fn):
    @functools.wraps(value_fn)
    def val_and_grad(x):
        return tfp.math.value_and_gradient(value_fn, x)
    return val_and_grad

@contextlib.contextmanager
def timed_execution():
    t0 = time.time()
    yield
    dt = time.time() - t0
    print('Evaluation took: %f seconds' % dt)


def np_value(tensor):
    """Get numpy value out of possibly nested tuple of tensors."""
    if isinstance(tensor, tuple):
        return type(tensor)(*(np_value(t) for t in tensor))
    elif isinstance(tensor, list):
        return [t.numpy()[0] for t in tensor]
    else:
        return tensor.numpy()

def run(optimizer):
    """Run an optimizer and measure it's evaluation time."""
    optimizer()
    with timed_execution():
        result = optimizer()
    return np_value(result)

class Results():
    def __init__(self):
        position = []

        
class HistoryLogger(tf.keras.callbacks.Callback):
    def __init__(self):
        super(HistoryLogger, self).__init__()
        self.history = {}
    
    def on_train_begin(self, logs=None):
        self.history = {}
    
    def on_epoch_begin(self, epoch, logs=None):
        for key, value in logs.items():
            self.history.setdefault(key, []).append(value)
    
    def on_epoch_end(self, epoch, logs=None):
        for key, value in logs.items():
            self.history.setdefault(key, []).append(value)
    
    def on_train_end(self, logs=None):
        if logs.get('plot_loss', False):
            plt.plot(np.arange(len(self.history['loss_train'])), self.history['loss_train'], label='train loss')
        if logs.get('plot_f2_train', False):
            plt.plot(np.arange(len(self.history['f2_train'])), self.history['f2_train'], label='train f2')
        if logs.get('plot_f2_val', False):
            plt.plot(np.arange(len(self.history['f2_val'])), self.history['f2_val'], label='validation f2')
            plt.legend()
        
history = HistoryLogger()

In [60]:
def generate_avail_data(X_tensor):
    avail_vals = []
    for cat in output_categories:
        if cat in avail_variable_names:
            avail_vals.append(X_tensor[:, avail_variable_indexer[cat]])
            avail_vals[-1] = avail_vals[-1].astype(np.float32)
        else:
            avail_vals.append(tf.ones(X_tensor.shape[0], dtype='float32'))
    return tf.stack(avail_vals)



def create_alt_spec_params(alternatives, name, gu_values, val_count):
    """
    alternatives (list of str): Names of various corresponding to which this variable should be created.
    name (str): The name of this variable.
    """
    alt_spec_vars = []
    if name == 'nearest_bus_dist':
        alternative='bus'
        alt_spec_var = tf.Variable([gu_values[val_count]], dtype='float32', 
                        name=f'{name}_{alternative}', trainable=True)
        val_count += 1
        alt_spec_vars.append(alt_spec_var)
        
    elif name == 'nearest_train_dist':
        alternative='train'
        alt_spec_var = tf.Variable([gu_values[val_count]], dtype='float32', 
                        name=f'{name}_{alternative}', trainable=True)
        val_count += 1
        alt_spec_vars.append(alt_spec_var)
        
    else:
        for alternative in alternatives:
            alt_spec_var = tf.Variable([gu_values[val_count]], dtype='float32', 
                            name=f'{name}_{alternative}', trainable=True)
            val_count += 1
            alt_spec_vars.append(alt_spec_var)
    return alt_spec_vars, val_count



def initiate_params(alt_spec_vars=True, ind_spec_vars_list=None):
    model_params = []
    
    gu_values = tf.abs(tf.keras.initializers.glorot_uniform(seed=1)(shape=(200,))*0)
    val_count = 0
    
    if alt_spec_vars:
        # Generic parameters for alternative specific variables of travelcost, traveltime, travelincentive
        gen_tc = tf.Variable([gu_values[val_count]], dtype=tf.float32, name='travelcost', trainable=True)
        val_count += 1
        gen_tt = tf.Variable([gu_values[val_count]], dtype=tf.float32, name='traveltime', trainable=True)
        val_count += 1
#         gen_ti = tf.Variable([gu_values[val_count]], dtype=tf.float32, name='travelincentive', trainable=True)
#         val_count += 1
        model_params.extend([gen_tc, gen_tt])

        # Alternative specific constants for bus, train, walk, bike, motor
        asc_bus = tf.Variable([gu_values[val_count]], dtype=tf.float32, name='bus_constant', trainable=True)
        val_count += 1
        asc_train = tf.Variable([gu_values[val_count]], dtype=tf.float32, name='train_constant', trainable=True)
        val_count += 1
        asc_walk = tf.Variable([gu_values[val_count]], dtype=tf.float32, name='walk_constant', trainable=True)
        val_count += 1
        asc_bike = tf.Variable([gu_values[val_count]], dtype=tf.float32, name='bike_constant', trainable=True)
        val_count += 1
        asc_motor = tf.Variable([gu_values[val_count]], dtype=tf.float32, name='motor_constant', trainable=True)
        val_count += 1
        model_params.extend([asc_bus, asc_train, asc_walk, asc_bike, asc_motor])
    
    if ind_spec_vars_list is not None:
        alternatives = ['bus', 'train', 'walk', 'bike', 'motor']
        for var in ind_spec_vars_list:
            isv_asp_params, val_count = create_alt_spec_params(alternatives, var, gu_values, val_count)
            model_params.extend(isv_asp_params)
    
    return model_params


def utilities(x_name, x_val, params, alt_spec_vars=True, ind_spec_vars_list=None):
    x_val = tf.convert_to_tensor(x_val, dtype='float32')
    ntrips = x_val.shape[-1]
    X = dict()
    x_val_trans = tf.transpose(x_val)
    for i, name in enumerate(x_name):
        val = x_val_trans[i]
        X[name] = val
            
    if alt_spec_vars:
        # Contribution to utility from tc, tt, ti
        travel_util_bus = X['buscost'] * params[0] + X['bustime'] * params[1] + params[2]
        travel_util_train = X['traincost'] * params[0] + X['traintime'] * params[1] + params[3]
        travel_util_walk = X['walktime'] * params[1] + params[4]  # walkcost is 0
        travel_util_bike = X['biketime'] * params[1] + params[5]  # bikecost is 0
        travel_util_motor = X['motorcost'] * params[0] + X['motortime'] * params[1] + params[6]
        travel_util_car = X['carcost'] * params[0] + X['cartime'] * params[1]

        utility_bus = travel_util_bus
        utility_car = travel_util_car
        utility_train = travel_util_train
        utility_walk = travel_util_walk
        utility_bike = travel_util_bike
        utility_motor = travel_util_motor

    if ind_spec_vars_list is not None:
        nparams = len(params)
        param_count = 0
        var_count = 0
        for i in range(7, nparams):
            isv_asp_param = params[i]
            var_name = ind_spec_vars_list[var_count]
            isv_util = X[var_name] * isv_asp_param
            
            if var_name == 'nearest_bus_dist':
                utility_bus += isv_util
                var_count += 1
                
            elif var_name == 'nearest_train_dist':
                utility_train += isv_util
                var_count += 1
        
            else:
                utility_bus += tf.where(tf.equal(param_count, 0), isv_util, 0.0)
                utility_train += tf.where(tf.equal(param_count, 1), isv_util, 0.0)
                utility_walk += tf.where(tf.equal(param_count, 2), isv_util, 0.0)
                utility_bike += tf.where(tf.equal(param_count, 3), isv_util, 0.0)
                utility_motor += tf.where(tf.equal(param_count, 4), isv_util, 0.0)
                
                param_count += 1
                param_count = param_count % 5
                if param_count == 0:
                    var_count += 1

    return tf.transpose(tf.stack((utility_bus, utility_car, utility_train, utility_walk, utility_bike, utility_motor)))


In [61]:
def initialize_layers(activation_layer, layers=0, nodes=[6], seed=1):
    non_linear_layers = []
    for i in range(layers):
        non_linear_layers.append(tf.keras.layers.Dense(
                                    nodes[i], 
                                    activation=activation_layer, 
                                    kernel_initializer = tf.keras.initializers.GlorotUniform(seed=seed),
                                    bias_initializer = tf.keras.initializers.GlorotUniform(seed=seed)
                                ))
    return non_linear_layers



def get_logits(x_name, x_val, params, alt_spec_vars=True, ind_spec_vars_list=None, layers_list = [], debug=False):
    utils = utilities(x_name, x_val, params, alt_spec_vars=alt_spec_vars, 
                      ind_spec_vars_list=ind_spec_vars_list)
    if debug:
        print("utils:", utils)
        print("-----------------------")
    
    for i, dense_layer in enumerate(layers_list):
        utils = dense_layer(utils)
        if debug:
            print(f"Non-linear layer {i}:", utils)
            print("------------------------------")
    
    return utils


def loss(true_y, logits, avail_vals, null_loglike=False):
    initial_probs = tf.nn.softmax(logits)
    masked_probs = initial_probs*tf.transpose(avail_vals)
    final_probs = tf.divide(masked_probs, 
                            tf.reshape((tf.reduce_sum(masked_probs, axis=-1) + tf.keras.backend.epsilon()), 
                                       [-1, 1])
                           )
    if null_loglike:
        final_probs = tf.where(final_probs!=0, 1, final_probs)
        final_probs = tf.divide(final_probs,
                                tf.reshape(tf.reduce_sum(final_probs, axis=-1)+tf.keras.backend.epsilon(),
                                          [-1, 1])
                               )
    ce_loss = -tf.reduce_mean(tf.reduce_sum(true_y*tf.math.log(
        final_probs + tf.keras.backend.epsilon()), axis=-1))
    return ce_loss


def apply_regularization(loss, params, l1_lambda=0.001, l2_lambda=0.001):
    # Applying l2 penalty on parameters of alternative specific variables
    l2_penalty = l2_lambda*tf.reduce_sum(tf.math.square(params[:3]))
    
    # Applying l1 penalty on paramters of individual specific variables
    l1_penalty = l1_lambda*tf.reduce_sum(tf.math.abs(params[8:]))
    
    return loss + l1_penalty + l2_penalty


def predict(x_name, x_val, params, avail_vals, alt_spec_vars=True, ind_spec_vars_list=None, layers_list=[]):
    """
    Returns the final probabilites and mode predictions
    """
    utils = get_logits(x_name, x_val, params, alt_spec_vars=True, 
                       ind_spec_vars_list=ind_spec_vars_list, layers_list=layers_list)
    initial_probs = tf.nn.softmax(utils)
    masked_probs = initial_probs*tf.transpose(avail_vals)
    final_probs = tf.divide(masked_probs, 
                            tf.reshape((tf.reduce_sum(masked_probs, axis=-1) + tf.keras.backend.epsilon()), 
                                       [-1, 1])
                           )
    
    return final_probs, tf.argmax(final_probs, axis=1)

## Training

In [76]:
# Optimizer functions for BFGS method
def mnl_bfgs(l1_lambda, l2_lambda):
    
    @make_val_and_grad_fn
    def loss_val_nd_loss_grad(params):
        curr_utils = utilities(x_names, x_vals, params, ind_spec_vars_list=ind_spec_vars)
        curr_loss = loss(ohe_true_y, curr_utils, avail_vals)
        penalty_loss = apply_regularization(curr_loss, params, l1_lambda=l1_lambda, l2_lambda=l2_lambda)
        return penalty_loss
    
    def mnl_with_bfgs():
        return tfp.optimizer.bfgs_minimize(loss_val_nd_loss_grad, 
                                          initial_position=tf.reshape(params, shape=[-1]),
                                          tolerance=1e-8,
                                          max_iterations=1000)
    return mnl_with_bfgs()
        

def outer_mnl_bfgs(l2_lambda, l1_lambda, return_res=False):
    l1_lambda = tf.convert_to_tensor(l1_lambda, dtype=tf.float32)
    l2_lambda = tf.convert_to_tensor(l2_lambda, dtype=tf.float32)
    
    result_tensor = mnl_bfgs(l1_lambda, l2_lambda)
    
    results = np_value(result_tensor)
    y_pred = predict(x_names, x_vals, results.position, avail_vals, ind_spec_vars_list=ind_spec_vars)[1].numpy() + 1
    
    if return_res:
        return results
    
    return fbeta_score(y_oCBD, y_pred, beta=2, average='weighted')


In [77]:
ind_spec_vars = ['AGE', 'Female', 'high_income', 'nearest_bus_dist', 'nearest_train_dist',
                 'Early_morning_departure','AM_peak_departure', 'Off_peak_departure', 
                 'PM_peak_departure',
                 'Purpose_101', 'Purpose_998', 'Purpose_100', 
                 'Purpose_300', 'Purpose_200', 'Purpose_400', 'Purpose_600', 
                 'Purpose_500', 'Purpose_202', 'Purpose_201',
                 'job_type_Part time job', 'job_type_Management executive', 
                 'job_type_civil servant', 'job_type_Housewife',
                 'job_type_Self employed/ Freelance', 'job_type_Others', 
                 'job_type_Unemployed']
#ind_spec_vars= ['nearest_bus_dist', 'nearest_train_dist']
ind_spec_vars = []

config_dict = {
    'Training Data': 'Outside CBD',
    'Testing Data': 'Inside CBD',
    'param intialization': 'zero initialization',
    'evaluation metric': 'F2',
    'ind_spec_vars': ind_spec_vars,
    'lambda1': 0,
    'lambda2': 0
}

In [78]:
initiate_params(ind_spec_vars_list=ind_spec_vars)

[<tf.Variable 'travelcost:0' shape=(1,) dtype=float32, numpy=array([0.], dtype=float32)>,
 <tf.Variable 'traveltime:0' shape=(1,) dtype=float32, numpy=array([0.], dtype=float32)>,
 <tf.Variable 'bus_constant:0' shape=(1,) dtype=float32, numpy=array([0.], dtype=float32)>,
 <tf.Variable 'train_constant:0' shape=(1,) dtype=float32, numpy=array([0.], dtype=float32)>,
 <tf.Variable 'walk_constant:0' shape=(1,) dtype=float32, numpy=array([0.], dtype=float32)>,
 <tf.Variable 'bike_constant:0' shape=(1,) dtype=float32, numpy=array([0.], dtype=float32)>,
 <tf.Variable 'motor_constant:0' shape=(1,) dtype=float32, numpy=array([0.], dtype=float32)>]

In [79]:
def initialise_model_vars(X_df, y_arr):
    params = initiate_params(ind_spec_vars_list=ind_spec_vars)
    ohe_true_y = tf.one_hot(y_arr-1, 6, on_value=1, off_value=0, axis=-1, dtype='float32')
    x_names = X_df.columns
    x_vals = X_df.values
    avail_vals = generate_avail_data(x_vals)
    return x_names, x_vals, avail_vals, ohe_true_y, params

In [80]:
x_names, x_vals, avail_vals, ohe_true_y, params = initialise_model_vars(X_oCBD, y_oCBD)

In [82]:
%%time
results = outer_mnl_bfgs(0, 0, return_res=True)

Wall time: 9.27 s


In [84]:
print("BFGS Results")
print("Converged:", results.converged)
print("Iterations:", results.num_iterations)

std_err = np.sqrt(np.diag(pd.DataFrame(results.inverse_hessian_estimate)))/np.sqrt(len(y_oCBD))
t_ratio = results.position/std_err

res_df = pd.DataFrame({'Variable': [param.name[:-2] for param in params], 
              'Coef': results.position,
             'Std.err': std_err,
             't-ratio': t_ratio})
res_df

BFGS Results
Converged: True
Iterations: 22


,Variable,Coef,Std.err,t-ratio
0,travelcost,-1.73e-03,2.51e-04,-6.89
1,traveltime,-4.83e-02,3.77e-03,-12.80
2,bus_constant,-3.90e-01,9.26e-02,-4.22
3,train_constant,-6.14e-01,8.67e-02,-7.08
4,walk_constant,1.44e-01,1.22e-01,1.18
5,bike_constant,-2.63e-01,3.67e-02,-7.19
6,motor_constant,-9.82e-01,1.56e-01,-6.29


In [85]:
result_dir_path = 'res/cg/cost_time_only_model'

if not os.path.exists(result_dir_path):
    os.makedirs(result_dir_path)

model_note = "Trained using only travel cost and travel time via BFGS on Outside CBD data."

model_params_file_path = os.path.join(result_dir_path, 'model_params.csv')
if os.path.exists(model_params_file_path):
    os.remove(model_params_file_path)
res_df.to_csv(model_params_file_path, index=False)

model_note_file_path = os.path.join(result_dir_path, 'model_note.txt')
if os.path.exists(model_note_file_path):
    os.remove(model_note_file_path)
with open(model_note_file_path, 'w') as f:
    f.write(model_note)